# TMT data treatment

This code will work with the TMT data provided by the FGCZ.

Initial working file for each experiment (hTERT_HME1_1, hTERT_HME1_2, HEK293T_1) is the raw_abundances_matrix

**First step is to rename the files to have a unified labeling system** -> (CellLine)_ (Data:Type)_ (Treatment)_ (TimePoint)_ (Replicate)

All the data transformation and statistics I am going to do starting from that initial "raw_data" file.



In [1]:
from pandas import read_csv

from src.transformations import *  # all necessary pakages are imported within src.transformations

# hme1_1 = pd.read_csv("../../data/hme1_1_raw_sample.tsv", sep="\t")
# hme1_2 = pd.read_csv("../../data/hme1_2_raw_sample.tsv", sep="\t")
# hek_1 = pd.read_csv("../../data/hek_1_raw_sample.tsv", sep="\t")
#
# hme1_1 = pd.read_csv("../../Experiment/hme1_1/Data/All/20260427_hTERT_HME1_1.tsv", sep="\t") # contain raw abs
# hme1_2 = pd.read_csv("../../Experiment/hme1_2/Data/All/20260427_hTERT_HEM1_2.tsv", sep="\t") # contain raw abs
# hek_1 = pd.read_csv("../../Experiment/hek_1/Data/All/20260417_HEK293T_raw_selection.tsv", sep="\t") # contain raw abs

## Data transformations

Run the full transformation pipeline on each dataset:
- n:reps  # important to double check this number of reps identified for the LFQ dataset
- raw:mean / raw:median / raw:sd / raw:cv
- log2:abs (zeros treated as NaN)
- log2:mean / log2:median / log2:sd
- log2:FC (fold change vs. starve)
- log2:scaled (max-normalised fold change, amplitude between -1 and 1)
- log2:zscore (per-site z-score of the temporal profile, computed per condition per cell line across the time series; removes amplitude, keeps shape)

**No statistics are computed here.** p-values and FDR (previously `log2:pvalue` / `log2:FDR` /
`log2:adjustedFDR`, Welch t-test + Benjamini-Hochberg) have been removed from the pipeline —
differential testing is done downstream with the **limma** package in R, which gives moderated
t / F statistics and lets the plex be modelled as a blocking factor. The corresponding functions
(`compute_pvalues`, `compute_fdr`, `compute_log10_fdr`) are kept commented out in
`src/transformations.py` for reference.


In [2]:
hme1_1_transformed = run_all_transformations(hme1_1, cell_lines=["WT"], data_type="raw:abs", conditions=['_EGF_', '_INS_', '_EGFnINS_'])
hme1_2_transformed = run_all_transformations(hme1_2, cell_lines=["WT"], data_type="raw:abs", conditions=['_EGF_', '_INS_', '_EGFnINS_'])
hek_1_transformed  = run_all_transformations(hek_1,  cell_lines=["WT"], data_type="raw:abs", conditions=['_EGF_', '_INS_', '_EGFnINS_'])

print(f"hme1_1: {hme1_1.shape} -> {hme1_1_transformed.shape}")
print(f"hme1_2: {hme1_2.shape} -> {hme1_2_transformed.shape}")
print(f"hek_1:  {hek_1.shape}  -> {hek_1_transformed.shape}")

[WT] Found 3 conditions, 21 (treatment, timepoint) groups.
[WT] Done. Added 295 columns.

All cell lines processed. Total columns: 90 -> 385
[WT] Found 3 conditions, 21 (treatment, timepoint) groups.
[WT] Done. Added 295 columns.

All cell lines processed. Total columns: 106 -> 401
[WT] Found 3 conditions, 21 (treatment, timepoint) groups.
[WT] Done. Added 295 columns.

All cell lines processed. Total columns: 106 -> 401
hme1_1: (35002, 90) -> (35002, 385)
hme1_2: (50002, 106) -> (50002, 401)
hek_1:  (36567, 106)  -> (36567, 401)


In [3]:
# Quick check of the new per-site z-score columns (log2:zscore) produced by run_all_transformations.
# Each condition's time series is standardised independently per site: mean ~ 0, std ~ 1 across timepoints.
from src.column_spec import ColumnSpec

_z_egf = ColumnSpec.select(hme1_2_transformed, cell_lines=["WT"], data_type="log2:zscore", conditions=["_EGF_"])
print("EGF z-score columns:", _z_egf)
hme1_2_transformed[_z_egf].head()

EGF z-score columns: ['WT_log2:zscore_EGF_full', 'WT_log2:zscore_EGF_starve', 'WT_log2:zscore_EGF_2', 'WT_log2:zscore_EGF_5', 'WT_log2:zscore_EGF_10', 'WT_log2:zscore_EGF_15', 'WT_log2:zscore_EGF_90']


,WT_log2:zscore_EGF_full,WT_log2:zscore_EGF_starve,WT_log2:zscore_EGF_2,WT_log2:zscore_EGF_5,WT_log2:zscore_EGF_10,WT_log2:zscore_EGF_15,WT_log2:zscore_EGF_90
0,1.719098,-0.366660,-1.061852,-0.433478,0.966686,-1.224618,0.400823
1,2.250126,-0.865472,-0.914893,-0.237049,0.310669,-0.430627,-0.112755
2,-1.291940,-0.102210,-0.074980,1.774072,-0.179459,-1.094103,0.968620
3,-0.312939,1.921967,-1.493313,0.544728,-0.107268,-0.784965,0.231790
4,-1.264736,-0.489006,-0.733860,1.633720,1.217400,-0.632087,0.268568


### Step size between consecutive timepoints (`log2:step`)

`log2:FC` compares every timepoint against the **same** reference (starve), so a site that rises
early and then stays up keeps a large FC at every later timepoint even though nothing more is
happening. `log2:step` asks the complementary question — *what changed during this interval?*

    log2:step(t_i) = log2:FC(t_i) − log2:FC(t_(i-1))

with starve as the first reference, e.g. `WT_log2:step_EGF_2 = WT_log2:FC_EGF_2 − WT_log2:FC_EGF_starve`
and `WT_log2:step_EGF_5 = WT_log2:FC_EGF_5 − WT_log2:FC_EGF_2`.

- Columns exist **only for the timepoints after starve** — there is no `log2:step` for `full` or for
  `starve` itself. `full` is a separate media control, not the timepoint preceding starve, so it is
  left out of the chain entirely (`exclude_full=True`).
- Since `log2:FC_*_starve` is identically 0, the first stimulation timepoint reproduces its own
  `log2:FC`. The baseline column is subtracted explicitly rather than assumed to be 0.
- The steps telescope: summing all steps of a condition returns the last `log2:FC` of that condition.
- ⚠️ The intervals are **unequal** (2, 5, 10, 15, 90 min), so a step is an increment *per interval*,
  not a rate per minute. Divide by the interval width if a rate is what is wanted.

**Missing values.** A step joins two timepoints, so one missing `log2:FC` blanks **two** step columns
(the step into that timepoint and the step out of it) and leaves the site unusable for step-based
analysis. `drop_incomplete_fc_sites()` below reports these sites and removes them, but only while
there are at most `MAX_DROPPED_SITES` (10) of them — the known sparse `n:reps == 1` tail. Above that
threshold it drops nothing and warns instead, so a systematic problem is not quietly deleted.

Counts measured on the 2026-08-07 processed tables: **hme1_2 → 4 sites** (all `n:reps == 1`, one
missing timepoint each) and **hme1_1 → 3**, both dropped; **hek_1 → 20**, above the guard, so its
call is left commented out until the cause is checked.


In [2]:
# Standalone entry point: load the finished table so the step-size cell below can be run
# on its own, without re-running the transformation / limma / PhosphoSitePlus cells above.
# low_memory=False silences the DtypeWarning on the sparse annotation columns (no .fillna —
# this table carries limma statistics, where NaN must stay NaN).
hme1_2_transformed = pd.read_csv("../../Experiment/hme1_2/Data/Processed/20260807_hTERT_HEM1_2_processed_phPlus.tsv", sep="\t", low_memory=False)


In [3]:
from src.filters import filter_incomplete_timeseries

CONDITIONS = ['_EGF_', '_INS_', '_EGFnINS_']
MAX_DROPPED_SITES = 10   # above this, a hole is a data problem — look before deleting rows


# def drop_incomplete_fc_sites(df,
#                              dataset_name,
#                              cell_line="WT",
#                              conditions=CONDITIONS,
#                              max_dropped=MAX_DROPPED_SITES,):
#     """
#     Report, and if they are few enough drop, the sites whose log2:FC time series has a hole.
#
#     log2:step is a difference between consecutive timepoints, so a single missing log2:FC
#     value blanks TWO step columns (the step into that timepoint and the step out of it) and
#     makes the site unusable for any step-based analysis. Such holes are sites that were not
#     quantified at one timepoint at all — in practice the sparse n:reps == 1 tail of the TMT
#     data, where a peptide has no reporter intensity in any replicate of that channel.
#
#     Sites are dropped only when there are at most `max_dropped` of them, i.e. when this is
#     the known sparse tail and not something systematic. Above that threshold nothing is
#     removed and a warning is printed, so the cause is looked at before rows are deleted.
#
#     Args:
#         df: transformed DataFrame carrying log2:FC columns.
#         dataset_name: label used in the printed report, e.g. 'hme1_2'.
#         cell_line: cell line prefix of the columns to check, e.g. 'WT'.
#         conditions: list of condition substrings whose series must be complete.
#         max_dropped: maximum number of sites that may be dropped without inspection.
#
#     Returns:
#         DataFrame with the incomplete sites removed, or the input unchanged if there are
#         none or if there are more than `max_dropped` of them.
#     """
#     fc_cols = ColumnSpec.select(df,
#                                 cell_lines=[cell_line],
#                                 data_type="log2:FC",
#                                 conditions=conditions,
#                                 exclude_full=True,)
#     incomplete = df[fc_cols].isna().any(axis=1)
#     n_incomplete = int(incomplete.sum())
#     print(f"[{dataset_name}] sites with a hole in the log2:FC series: "
#           f"{n_incomplete} / {len(df)}")
#
#     if n_incomplete == 0:
#         return df
#     print(df.loc[incomplete, ["site", "n:reps"]].to_string(index=False))
#
#     if n_incomplete > max_dropped:
#         print(f"[{dataset_name}] more than {max_dropped} incomplete sites — NOTHING DROPPED. "
#               f"Check the cause before removing rows (raise MAX_DROPPED_SITES to drop anyway); "
#               f"their log2:step values stay NaN.")
#         return df
#
#     filtered = filter_incomplete_timeseries(df,
#                                             cell_lines=[cell_line],
#                                             conditions=conditions,
#                                             data_type="log2:FC",
#                                             exclude_full=True,
#                                             max_missing=0,)
#     print(f"[{dataset_name}] dropped {len(df) - len(filtered)} site(s) -> {len(filtered)} rows")
#     return filtered


# hme1_1_transformed = drop_incomplete_fc_sites(hme1_1_transformed, "hme1_1")
# hme1_1_transformed = log2_step_size(hme1_1_transformed,
#                                     cell_line="WT",
#                                     conditions=CONDITIONS,)
# hme1_2_transformed = drop_incomplete_fc_sites(hme1_2_transformed, "hme1_2")
hme1_2_transformed = log2_step_size(hme1_2_transformed,
                                    cell_line="WT",
                                    conditions=CONDITIONS,)
# hek_1_transformed = drop_incomplete_fc_sites(hek_1_transformed, "hek_1")   # 20 incomplete sites — above the guard
# hek_1_transformed = log2_step_size(hek_1_transformed,
#                                    cell_line="WT",
#                                    conditions=CONDITIONS,)

_step_egf = ColumnSpec.select(hme1_2_transformed,
                              cell_lines=["WT"],
                              data_type="log2:step",
                              conditions=["_EGF_"],)
print("EGF step columns:", _step_egf)

# Sanity check: the steps telescope back to the last fold change of the condition.
# Compared on the shared index — dropna() on each side separately would compare two
# different sets of sites by position whenever any value is missing.
_step_sum = hme1_2_transformed[_step_egf].sum(axis=1, skipna=False)
_fc_last = hme1_2_transformed["WT_log2:FC_EGF_90"]
_comparable = _step_sum.notna() & _fc_last.notna()
print(f"sum(steps) == log2:FC at 90 min: "
      f"{np.allclose(_step_sum[_comparable], _fc_last[_comparable])} "
      f"({(~_comparable).sum()} site(s) skipped: incomplete series)")

hme1_2_transformed[["WT_log2:FC_EGF_starve", "WT_log2:FC_EGF_2", "WT_log2:FC_EGF_5"] + _step_egf].head()


EGF step columns: ['WT_log2:step_EGF_2', 'WT_log2:step_EGF_5', 'WT_log2:step_EGF_10', 'WT_log2:step_EGF_15', 'WT_log2:step_EGF_90']
sum(steps) == log2:FC at 90 min: True (3 site(s) skipped: incomplete series)


,WT_log2:FC_EGF_starve,WT_log2:FC_EGF_2,WT_log2:FC_EGF_5,WT_log2:step_EGF_2,WT_log2:step_EGF_5,WT_log2:step_EGF_10,WT_log2:step_EGF_15,WT_log2:step_EGF_90
0,0.0,-0.350936,-0.033730,-0.350936,0.317206,0.706809,-1.106180,0.820530
1,0.0,-0.030278,0.385017,-0.030278,0.415295,0.335571,-0.454171,0.194751
2,0.0,0.003562,0.245424,0.003562,0.241862,-0.255528,-0.119638,0.269811
3,0.0,-1.421528,-0.573243,-1.421528,0.848286,-0.271378,-0.282075,0.423200
4,0.0,-0.031538,0.273416,-0.031538,0.304955,-0.053624,-0.238222,0.116008


In [4]:
hme1_2_transformed.to_csv("../../Experiment/hme1_2/Data/Processed/20260817_hTERT_HEM1_2_log2_processed.tsv", sep="\t", index=False)

In [4]:
# Save transformed datasets

# hek_1_transformed.to_csv("../../Experiment/hek_1/Data/Processed/20260807_HEK293T_log2_processed.tsv", sep="\t", index=False)
# hme1_1_transformed.to_csv("../../Experiment/hme1_1/Data/Processed/20260807_hTERT_HEM1_1_log2_processed.tsv", sep="\t", index=False)
# hme1_2_transformed.to_csv("../../Experiment/hme1_2/Data/Processed/20260807_hTERT_HEM1_2_log2_processed.tsv", sep="\t", index=False)

# hek_1_transformed.to_csv("../../data/hek_1_log2_sample.tsv", sep="\t", index=False)
# hme1_1_transformed.to_csv("../../data/hme1_1_log2_sample.tsv", sep="\t", index=False)
# hme1_2_transformed.to_csv("../../data/hme1_2_log2_sample.tsv", sep="\t", index=False)

## Statistics — merging the limma results

The p-values are **not** computed here. `limma_for_pvalues.rmd` (this folder, R) reads the
`*_log2_sample.tsv` files written by the cell above, fits a blocked linear model on `log2:abs`
and writes one `data/{dataset}_limma_pvalues.tsv` per dataset, keyed by `site`.

**Run that R notebook before this cell.** It must be re-run whenever the transformation step
changes, since it reads the files saved above.

Why R: with 4 replicates the per-site variance has 3 degrees of freedom and is very unstable.
limma shrinks it toward a prior fitted across all sites (moderated *t* / *F*, Smyth 2004), and
lets the TMT plex enter as a blocking factor — each timepoint is compared to the starve control
**of its own plex**, which is how the experiment was actually run.

Columns merged in, each contrasted against `starve`:

| Column | Meaning |
|---|---|
| `WT_log2:limmaFC_EGF_5` | log2 fold change from the blocked model |
| `WT_log2:pvalue_EGF_5` | moderated *t* p-value |
| `WT_log2:FDR_EGF_5` | Benjamini-Hochberg FDR across sites |
| `WT_log2:adjustedFDR_EGF_5` | `-log10(FDR)` |
| `WT_log2:Fpvalue_EGF_omnibus` | omnibus *F*: does this site differ from starve at **any** EGF timepoint? (also `INS`, `EGFnINS`, and `ALL` across every condition) |
| `WT_log2:FFDR_..._omnibus`, `WT_log2:adjustedFFDR_..._omnibus` | FDR and `-log10(FDR)` of that *F*-test |

Two things to keep in mind when using these numbers:

- **Sites limma did not test carry NaN.** It only fits sites detected in at least 2 plexes
  (`MIN_PLEX`), because a site seen in a single plex has zero residual degrees of freedom and
  its p-value would come entirely from the prior.
- **No between-sample normalisation was applied** before the fit, so the fold changes carry the
  loading shift documented in `clustering_method_decision.md` §1. The `NORMALIZE` parameter in
  the R notebook switches this on when that fix is made.


In [5]:
# Merge the limma statistics computed in R back onto the transformed datasets.
# Left join on "site": every row is kept, and sites limma did not test (detected in fewer
# plexes than its MIN_PLEX threshold) get NaN rather than being dropped.
# Re-running this cell is safe — a previous merge is replaced, not duplicated.
hek_1_transformed = pd.read_csv("../../Experiment/hek_1/Data/Processed/20260807_HEK293T_log2_processed.tsv", sep="\t")
hme1_1_transformed = pd.read_csv("../../Experiment/hme1_1/Data/Processed/20260807_hTERT_HEM1_1_log2_processed.tsv", sep="\t")
hme1_2_transformed = pd.read_csv("../../Experiment/hme1_2/Data/Processed/20260807_hTERT_HEM1_2_log2_processed.tsv", sep="\t")



hme1_1_transformed = merge_limma_results(hme1_1_transformed, "../../Experiment/hme1_1/Data/Processed/20260807_hTERT_HEM1_1_limma_pvalues.tsv")
hme1_2_transformed = merge_limma_results(hme1_2_transformed, "../../Experiment/hme1_2/Data/Processed/20260807_hTERT_HEM1_2_limma_pvalues.tsv")
hek_1_transformed  = merge_limma_results(hek_1_transformed, "../../Experiment/hek_1/Data/Processed/20260807_HEK293T_limma_pvalues.tsv")

print(f"\nhme1_1: {hme1_1_transformed.shape}")
print(f"hme1_2: {hme1_2_transformed.shape}")
print(f"hek_1:  {hek_1_transformed.shape}")

  merged 76 limma columns from ../../Experiment/hme1_1/Data/Processed/20260807_hTERT_HEM1_1_limma_pvalues.tsv
  25209 / 35002 sites carry statistics (9793 not tested by limma)
  merged 76 limma columns from ../../Experiment/hme1_2/Data/Processed/20260807_hTERT_HEM1_2_limma_pvalues.tsv
  33719 / 50002 sites carry statistics (16283 not tested by limma)
  merged 76 limma columns from ../../Experiment/hek_1/Data/Processed/20260807_HEK293T_limma_pvalues.tsv
  23282 / 36567 sites carry statistics (13285 not tested by limma)

hme1_1: (35002, 461)
hme1_2: (50002, 477)
hek_1:  (36567, 477)


In [6]:
# Inspect what was merged: the limma columns are grouped by data type, timepoints in order.
_limma_cols = [c for c in hme1_2_transformed.columns if any(
    s in c for s in ("limmaFC", "pvalue", "FDR", "Fpvalue", "FFDR"))]
print(f"{len(_limma_cols)} limma columns added, e.g.:")
for _c in _limma_cols[:6]:
    print("   ", _c)

# How many sites respond to EGF at any timepoint (omnibus F), and at 5 min specifically?
_f_col = "WT_log2:FFDR_EGF_omnibus"
_t_col = "WT_log2:FDR_EGF_5"
print(f"\nEGF-responsive at any timepoint (FDR < 0.05): {(hme1_2_transformed[_f_col] < 0.05).sum()}")
print(f"significant at EGF 5 min       (FDR < 0.05): {(hme1_2_transformed[_t_col] < 0.05).sum()}")
print(f"sites without statistics (not tested by limma): {hme1_2_transformed[_f_col].isna().sum()}")

hme1_2_transformed[["site", "n:reps", "WT_log2:limmaFC_EGF_5", "WT_log2:pvalue_EGF_5",
                    "WT_log2:FDR_EGF_5", "WT_log2:adjustedFDR_EGF_5",
                    "WT_log2:Fpvalue_EGF_omnibus"]].head()

76 limma columns added, e.g.:
    WT_log2:limmaFC_EGF_full
    WT_log2:limmaFC_EGF_2
    WT_log2:limmaFC_EGF_5
    WT_log2:limmaFC_EGF_10
    WT_log2:limmaFC_EGF_15
    WT_log2:limmaFC_EGF_90

EGF-responsive at any timepoint (FDR < 0.05): 20853
significant at EGF 5 min       (FDR < 0.05): 15630
sites without statistics (not tested by limma): 16283


,site,n:reps,WT_log2:limmaFC_EGF_5,WT_log2:pvalue_EGF_5,WT_log2:FDR_EGF_5,WT_log2:adjustedFDR_EGF_5,WT_log2:Fpvalue_EGF_omnibus
0,A0A2R8Y4L2_197_199_1_0~SGSGNFGGGR,1,NaN,NaN,NaN,NaN,NaN
1,A0A2R8Y4L2_197_199_1_1_S199~SGsGNFGGGR,3,0.385017,0.119692,0.174066,0.759286,0.030320
2,A0A2R8Y4L2_2_6_1_1_S6~SKSEsPKEPEQLR,3,0.245424,0.017072,0.040042,1.397489,0.009403
3,A0A2R8Y4L2_2_6_2_2_S2S4~sKsESPK,1,NaN,NaN,NaN,NaN,NaN
4,A0A2R8Y4L2_305_316_1_0~NQGGYGGSSSSSSYGSGR;NQGG...,4,0.273416,0.012588,0.032211,1.491997,0.017254


## Merging external data

PhosphoSite Plus dataset

In [7]:
functional_score_df = pd.read_csv("../../External_Data/Metadata/PhosphoSitePlus.tsv", sep="\t")
regulatory_sites = pd.read_csv("../../External_Data/Metadata/Phosphosite/Regulatory_sites.tsv", sep="\t")

print(f"functional_score_df columns: {functional_score_df.columns}")
print(f"regulatory_sites columns: {regulatory_sites.columns}")

functional_score_df columns: Index(['protein_Id', 'protein_name', 'prot_seq_position', 'aa', 'site',
       'functional_score', 'ms_lit', 'ERK_motif', 'ERK_ext_motif'],
      dtype='object')
regulatory_sites columns: Index(['GENE', 'protein_name', 'information', 'protein_Id', 'GENE_ID',
       'HU_CHR_LOC', 'ORGANISM', 'MOD_RSD', 'SITE_GRP_ID', 'SITE_+/-7_AA',
       'DOMAIN', 'ON_FUNCTION', 'ON_PROCESS', 'ON_PROT_INTERACT',
       'ON_OTHER_INTERACT', 'PMIDs', 'LT_LIT', 'MS_LIT', 'MS_CST', 'NOTES'],
      dtype='object')


In [12]:
df_with_extra_info = merge_functional_score(df = hme1_2_transformed,
                                            phosphosite_df = functional_score_df,
                                            ph_residue ="aa",
                                            ph_position = "prot_seq_position")
df_with_extra_info = merge_phosphoplus_info(df = df_with_extra_info,
                                            phosphosite_df = functional_score_df,
                                            ph_residue ="aa",
                                            ph_position = "prot_seq_position",
                                            adding_info = ["ERK_motif"])
df_with_extra_info = merge_phosphoplus_info(df = df_with_extra_info,
                                            phosphosite_df = regulatory_sites,
                                            ph_residue ="aa",
                                            ph_position = "prot_seq_position",
                                            adding_info = ["ORGANISM", "ON_FUNCTION", "ON_PROCESS", "ON_PROT_INTERACT", 'ON_OTHER_INTERACT'],
                                            regulatory_sites= True)
df_with_extra_info

,protein_Id,nrPeptides,description,protein_name,protein_length,nr_tryptic_peptides,peptide_index,peptide_seq,SequenceWindow,Start,...,WT_log2:adjustedFFDR_INS_omnibus,WT_log2:adjustedFFDR_EGFnINS_omnibus,WT_log2:adjustedFFDR_ALL_omnibus,functional_score,ERK_motif,ORGANISM,ON_FUNCTION,ON_PROCESS,ON_PROT_INTERACT,ON_OTHER_INTERACT
0,A0A2R8Y4L2,13,Heterogeneous nuclear ribonucleoprotein A1-lik...,HNRNPA1L3,320.0,18.0,A0A2R8Y4L2_197_199_1_0,SGSGNFGGGR,SSSQRGR.SGSGNFGGGR.GGGFGGN,197,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,A0A2R8Y4L2,13,Heterogeneous nuclear ribonucleoprotein A1-lik...,HNRNPA1L3,320.0,18.0,A0A2R8Y4L2_197_199_1_1_S199,SGsGNFGGGR,SSSQRGR.SGsGNFGGGR.GGGFGGN,197,...,4.952779,6.593809,7.483290,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,A0A2R8Y4L2,13,Heterogeneous nuclear ribonucleoprotein A1-lik...,HNRNPA1L3,320.0,18.0,A0A2R8Y4L2_2_6_1_1_S6,SKSEsPKEPEQLR,.SKSEsPKEPEQLR.KIFIGGI,2,...,0.386928,0.312728,1.785596,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,A0A2R8Y4L2,13,Heterogeneous nuclear ribonucleoprotein A1-lik...,HNRNPA1L3,320.0,18.0,A0A2R8Y4L2_2_6_2_2_S2S4,sKsESPK,.sKsESPK.EPEQIRK,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,A0A2R8Y4L2,13,Heterogeneous nuclear ribonucleoprotein A1-lik...,HNRNPA1L3,320.0,18.0,A0A2R8Y4L2_305_316_1_0,NQGGYGGSSSSSSYGSGR;NQGGYGGSSSSSSYGSGRRF,QYFAKPR.NQGGYGGSSSSSSYGSGR.RF;QYFAKPR.NQGGYGGS...,301,...,0.695945,2.138090,3.050544,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49997,Q9Y3T6,1,R3H and coiled-coil domain-containing protein ...,R3HCC1,440.0,20.0,Q9Y3T6_236_237_1_0,FGSTLQLDLEK,MVEMATR.FGSTLQLDLEK.GKESIIE,234,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49998,Q9Y546,1,Leucine-rich repeat-containing protein 42 OS=H...,LRRC42,428.0,23.0,Q9Y546_384_388_1_0,HEAISSQESKK,CHGPVIK.HEAISSQESKK.SKKRPFE,380,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49999,Q9Y592,1,Centrosomal protein of 83 kDa OS=Homo sapiens ...,CEP83,701.0,31.0,Q9Y592_698_699_1_1_S698,KQLEELGsSGE,IETTQRK.KQLEELGsSGE.,691,...,NaN,NaN,NaN,NaN,False,human,intracellular localization,cytoskeletal reorganization,NaN,NaN
50000,Q9Y5J5,1,Pleckstrin homology-like domain family A membe...,PHLDA3,127.0,9.0,Q9Y5J5_119_127_1_1_S119,QsLGTGTLVS,IQTVRAR.QsLGTGTLVS.,118,...,NaN,NaN,NaN,0.416132,False,NaN,NaN,NaN,NaN,NaN


In [13]:
# df_with_extra_info.to_csv("../../Experiment/hek_1/Data/Processed/20260807_HEK293T_processed_phPlus.tsv", sep="\t", index=False)
# df_with_extra_info.to_csv("../../Experiment/hme1_1/Data/Processed/20260807_hTERT_HEM1_1_processed_phPlus.tsv", sep="\t", index=False)
# df_with_extra_info.to_csv("../../Experiment/hme1_2/Data/Processed/20260807_hTERT_HEM1_2_processed_phPlus.tsv", sep="\t", index=False)

In [ ]:
# df_with_extra_info.loc[df_with_extra_info["protein_name"] == "EGFR"]